# PyTorch training

Companion notebook for the [PyTorch training](https://ddacs.readthedocs.io/en/latest/tutorials/pytorch/) tutorial. It is written to stand on its own: every step is annotated so the notebook reads top to bottom.

## Walkthrough

1. construct a `DDACSDataset` over a published view,
2. iterate one batch through a `DataLoader`,
3. add a process-parameter filter and an explicit sim_id allowlist, grounded in the Croissant manifest,
4. build the canonical train / val / test splits from the CSV `split` column,
5. enable per-shard shuffle and `set_epoch` for multi-epoch training,
6. inspect the partial-download behaviour (graceful skip versus mlcroissant abort),
7. stream a custom view built with `ddacs.add_view` via the `dataset=` kwarg.

## Assumptions

- This notebook lives in `notebooks/` of the repository. The data directory `./data` therefore resolves to `../data` from here.
- `ddacs download --small` has been run once from the repository root, so `../data/` contains `metadata.json`, `process_parameters.csv`, and the bundled sample zip.
- The PyTorch extra is installed: `pip install ddacs[torch]`.

In [ ]:
import ddacs
from ddacs.pytorch import DDACSDataset
from torch.utils.data import DataLoader
print('ddacs', ddacs.__version__)

from pathlib import Path
DATA_DIR = Path('../data')
sim_id   = 258864

ddacs 3.1.3


## 1. Construct a `DDACSDataset`

`DDACSDataset(view, data_dir, ...)` is a `torch.utils.data.IterableDataset` that streams records of a Croissant view. At construction time it:

1. loads the manifest (same code path as `ddacs.load`),
2. resolves the view's fields to HDF5 paths and JSONPath transforms,
3. scans `data_dir` for zip files and builds an index `sim_id -> local zip path`,
4. reads `process_parameters.csv` and applies the optional `sim_ids` allowlist and `where` predicate.

After construction it knows exactly which simulations are locally available; iteration silently skips ones whose zip is missing.

In [2]:
ds = DDACSDataset(view='springback-minimal', data_dir=DATA_DIR)
print('view:           ', ds.view)
print('field specs:    ', ds._field_specs)
print('total sim_ids:  ', len(ds._sim_ids), '(every sim in process_parameters.csv)')
print('locally indexed:', len(ds._h5_index), '(only these will actually stream)')

view:            springback-minimal
field specs:     {'op10_blank_node_displacement_forming': ('OP10/blank/node_displacement', 2), 'op10_blank_node_displacement_springback': ('OP10/blank/node_displacement', 3)}
total sim_ids:   32466 (every sim in process_parameters.csv)
locally indexed: 1 (only these will actually stream)


## 2. Iterate one batch through a `DataLoader`

`DDACSDataset` plugs straight into a `DataLoader` with no extra glue. The default `collate_fn` stacks each field along a new leading batch axis. With the small bundle only the sample simulation has a local zip, so the first batch contains one record; with the full release `batch_size=16` would fill normally.

In [3]:
loader = DataLoader(ds, batch_size=16, num_workers=0)
for batch in loader:
    for k, v in batch.items():
        print(f'  {k:50s} shape={tuple(v.shape)} dtype={v.dtype}')
    break

  op10_blank_node_displacement_forming               shape=(1, 11236, 3) dtype=torch.float64
  op10_blank_node_displacement_springback            shape=(1, 11236, 3) dtype=torch.float64


## 3. Filter via the Croissant manifest

Both filters run against `process_parameters.csv` rows **before** any zip is opened, so the IO scales with the surviving simulations rather than with the full 32 466.

The row keys are not magic: they come straight from the Croissant manifest. `metadata.json` declares `process_parameters.csv` as a `FileObject` and exposes its columns as the `process-parameters` `RecordSet`. The same fields are what `ds.records('process-parameters')` yields when iterating directly through `mlcroissant`. `DDACSDataset` simply consumes those rows at construction time and applies the predicate before any zip is touched.

### What `where` receives

`DDACSDataset` reads `process_parameters.csv` with `pandas` and runs `where` once per row via `df.apply(where, axis=1)`. The `row` argument is a `pandas.Series` whose index is the CSV column names, so you can read columns with either `row['split']` or `row.split`. Values come back as native Python types:

| Column | Type | Example |
|--------|------|---------|
| `index` | `int` | `16039` |
| `geometry` | `str` | `'rectangular'`, `'concave'`, `'convex'` |
| `curvature_radius`, `bottom_radius`, `wall_angle`, ... | `float` | `30.0` |
| `split` | `str` | `'train'`, `'val'`, `'test'` |
| `rddac` | `bool` | `False` |

The full list mirrors what the manifest publishes; the cell below prints it once so the available keys are explicit. Any predicate returning truthy keeps the row.

- `where=<callable>`: any function `pd.Series -> bool`.
- `sim_ids=[...]`: explicit allowlist of integers, applied before `where`.

Both can be combined; the predicate is applied after the allowlist.

In [4]:
# Inspect the manifest's process-parameters columns once.
raw = ddacs.load(data_dir=DATA_DIR)
sample = next(iter(raw.records('process-parameters')))
print('manifest columns:', [k.split('/')[-1] for k in sample.keys()])

rect = DDACSDataset(
    view='springback-minimal',
    data_dir=DATA_DIR,
    where=lambda row: row['geometry'] == 'rectangular',
)
print(f'rectangular-only sim_ids: {len(rect._sim_ids):>6d} (of 32466)')

ids_only = DDACSDataset(
    view='springback-minimal',
    data_dir=DATA_DIR,
    sim_ids=[sim_id],
)
print(f'sim_ids=[sim_id]:         {len(ids_only._sim_ids):>6d}')

manifest columns: ['index', 'geometry', 'curvature_radius', 'bottom_radius', 'wall_angle', 'material_scaling_factor', 'sheet_metal_thickness', 'friction_coefficient', 'blankholder_force', 'split', 'rddac']


rectangular-only sim_ids:  10689 (of 32466)


sim_ids=[sim_id]:              1


## 4. Train / val / test splits

`process_parameters.csv` ships with a `split` column whose canonical values are `'train'`, `'val'`, and `'test'`. Because the column is part of the Croissant manifest, the same `where=` predicate that filtered by geometry above works on it. Three `DDACSDataset` instances, one per split, is the fastest way to wire up a training loop without writing any custom partitioning code.

Shuffle the train split for SGD; leave `val`/`test` deterministic for reproducible evaluation.

In [5]:
splits = {}
for name in ('train', 'val', 'test'):
    splits[name] = DDACSDataset(
        view='springback-minimal',
        data_dir=DATA_DIR,
        where=lambda row, n=name: row['split'] == n,
        shuffle=(name == 'train'),
        seed=42,
    )

for name, split_ds in splits.items():
    streamable = sum(1 for sid in split_ds._sim_ids if sid in split_ds._h5_index)
    print(f"{name:>5s}: {len(split_ds._sim_ids):>6d} sim_ids (of 32466), "
          f"{streamable:>3d} streamable now (zip on disk)")

train:  25973 sim_ids (of 32466),   0 streamable now (zip on disk)
  val:   3246 sim_ids (of 32466),   1 streamable now (zip on disk)
 test:   3247 sim_ids (of 32466),   0 streamable now (zip on disk)


## 5. Shuffle + `set_epoch`

`shuffle=True` permutes each shard with a seed derived from `seed + epoch + shard_id`. Worker shards stay disjoint, so two workers do not produce the same simulation. Call `set_epoch(n)` once per epoch to get a different permutation each time. Without it, every epoch sees the same order, which biases optimisation.

In [6]:
ds_shuf = DDACSDataset(
    view='springback-minimal',
    data_dir=DATA_DIR,
    shuffle=True,
    seed=42,
)
for epoch in range(2):
    ds_shuf.set_epoch(epoch)
    print(f'epoch {epoch}: first sim_id this shard would visit -> {ds_shuf._sim_ids[0] if not ds_shuf.shuffle else "(shuffled per shard)"}')

epoch 0: first sim_id this shard would visit -> (shuffled per shard)
epoch 1: first sim_id this shard would visit -> (shuffled per shard)


## 6. Partial download: graceful skip vs `records()` abort

Two access paths exist; only one is friendly to partial downloads.

- **`ds.records(view)`** (the underlying mlcroissant call) walks every zip in the FileSet alphabetically and aborts the iteration at the first missing one. Useful when the full release is on disk.
- **`DDACSDataset`** uses the `sim_id -> local zip` index built at construction. Simulations whose zip is missing are silently skipped, so iteration on the small bundle still yields the bundled sample.

The cell below shows both side by side.

In [7]:
import numpy as np

# 1) mlcroissant records : expected to abort on the first missing zip.
raw = ddacs.load(data_dir=DATA_DIR)
yielded = 0
try:
    for rec in raw.records('springback-minimal'):
        yielded += 1
        break
except Exception as e:
    print(f'records(): yielded {yielded} record(s) then {type(e).__name__}: {str(e)[:120]}')

# 2) DDACSDataset : graceful skip.
yielded = sum(1 for _ in DDACSDataset(view='springback-minimal', data_dir=DATA_DIR))
print(f'DDACSDataset: yielded {yielded} record(s) without error')

records(): yielded 0 record(s) then GenerationError: An error occured during the sequential generation of the dataset, more specifically during the operation Download(106921


DDACSDataset: yielded 1 record(s) without error


## 7. Stream a custom view (`add_view` + `dataset=`)

The published views (`springback-minimal`, `forming-snapshot`, ...) cover the common targets, but the [Build your own view](https://ddacs.readthedocs.io/en/latest/tutorials/views/) tutorial shows how to compose a custom `RecordSet` with `ddacs.add_view`. `add_view` mutates the in-memory `mlcroissant.Dataset` you hand it; the published manifest on DaRUS stays unchanged.

To stream that custom view through `DDACSDataset`, pass the same loaded object via the `dataset=` kwarg. Without it, `DDACSDataset.__init__` would re-parse `metadata.json` from disk and the in-memory mutation would be invisible (it would raise `ValueError: view 'my-view' not found`). With it, the constructor uses the caller's object as-is.

In [8]:
ds_manifest = ddacs.load(data_dir=DATA_DIR)
ddacs.add_view(
    ds_manifest,
    'forming-only',
    fields={
        'forming': ('op10_blank_node_displacement', 2),
    },
)

custom_ds = DDACSDataset(
    view='forming-only',
    data_dir=DATA_DIR,
    dataset=ds_manifest,
)

print('view:           ', custom_ds.view)
print('field specs:    ', custom_ds._field_specs)
print('total sim_ids:  ', len(custom_ds._sim_ids))
print('locally indexed:', len(custom_ds._h5_index))

# One record through the loader: the custom view ships exactly one field.
loader = DataLoader(custom_ds, batch_size=4, num_workers=0)
for batch in loader:
    for k, v in batch.items():
        print(f'  {k:30s} shape={tuple(v.shape)} dtype={v.dtype}')
    break

view:            forming-only
field specs:     {'forming': ('OP10/blank/node_displacement', 2)}
total sim_ids:   32466
locally indexed: 1
  forming                        shape=(1, 11236, 3) dtype=torch.float64
